# L3-20：非 Ascon AXI GPIO 寄存器段

默认加载 PYNQ BaseOverlay，自动选择名称含 `led` 的 AXI GPIO。所有访问通过 interface
contract 和 `RegisterMap`，不硬编码地址。测试结束会恢复原始方向和值。

如板上 IP 名不同，可设置 `L3_GPIO_IP`，但非 LED 路径默认被拒绝，避免误驱动外部引脚。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import subprocess
import sys

import pynq


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


repo = Path("/home/xilinx/jupyter_notebooks/amd_fpga_2026").resolve()
assert (repo / "skill/ab_eval_pynq/test_board").is_dir(), (
    "fixed repository path does not exist: %s" % repo
)
os.chdir(str(repo))
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "skill/pynq-overlay-workflow/scripts"))

run_dir = (
    Path(os.environ["L3_RUN_DIR"]).resolve()  # 可选：不填时读取 l3_current_session.json
    if os.environ.get("L3_RUN_DIR")
    else Path(
        json.loads(
            (repo / "data/skill/l3_current_session.json").read_text(encoding="utf-8")
        )["run_dir"]
    ).resolve()
)
assert run_dir.is_dir(), run_dir
assert (repo / "data/skill").resolve() in run_dir.parents, (
    "L3_RUN_DIR must stay under data/skill"
)
assert (run_dir / "00_preflight.json").is_file(), "run l3_00_preflight.ipynb first"
run_id = run_dir.name


def base_record(case, claim_level):
    return {
        "case": case,
        "status": "RUNNING",
        "claim_level": claim_level,
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "run_id": run_id,
        "run_dir": str(run_dir),
        "hostname": platform.node(),
        "python_version": platform.python_version(),
        "pynq_version": pynq.__version__,
        "kernel": platform.platform(),
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
        ).strip(),
        "checks": [],
    }


def check(record, name, condition, detail=None):
    item = {"name": name, "status": "PASS" if condition else "FAIL", "detail": detail}
    record["checks"].append(item)
    assert condition, item

In [ ]:
import importlib
import time

from pynq import MMIO
from pynq.ps import Clocks
from pynq_skill import artifacts, hwh, overlay_check, regmap

record = base_record("non_ascon_gpio", "hardware_measured")  # 缺 BaseOverlay/GPIO 时会降为 not_measured
out_path = run_dir / "20_non_ascon_gpio.json"
record.update(
    {
        "simulated": False,  # BaseOverlay 是真实 PL 工件；未跑通时状态不会是 PASS
        "overlay_kind": "PYNQ BaseOverlay",
        "selected_ip": None,  # 自动选取 LED GPIO；也可用 L3_GPIO_IP 显式指定
        "gpio_width": None,  # 从 hwh/ip_dict 的 C_GPIO_WIDTH 自动读取
        "register_trace": [],  # 运行时逐条追加写值和读回值
        "hwh_cross_check": [],  # 运行时填入 contract 与 hwh 的差异说明
    }
)

try:
    base_module = importlib.import_module("pynq.overlays.base")
    package_dir = Path(base_module.__file__).resolve().parent
    bit_path = package_dir / "base.bit"
    hwh_path = package_dir / "base.hwh"
    if not bit_path.is_file() or not hwh_path.is_file():
        record["status"] = "NOT_RUN"
        record["claim_level"] = "not_measured"
        record["reason"] = "PYNQ BaseOverlay bit/hwh not found"
        record["package_dir"] = str(package_dir)
        write_json(out_path, record)
        raise RuntimeError(record["reason"])

    art = artifacts.check_artifacts(bit_path)
    info = hwh.parse_hwh(hwh_path)
    overlay = base_module.BaseOverlay(download=True)
    candidates = sorted(
        name
        for name, item in overlay.ip_dict.items()
        if str(item.get("type", "")).startswith("xilinx.com:ip:axi_gpio:")
    )
    selected = os.environ.get("L3_GPIO_IP")  # 通常留空；留空时自动找名称含 led 的 IP
    if selected is None:
        led_candidates = [name for name in candidates if "led" in name.lower()]
        selected = led_candidates[0] if led_candidates else None
    if selected is None:
        record["status"] = "NOT_RUN"
        record["claim_level"] = "not_measured"
        record["reason"] = "no LED AXI GPIO found; set L3_GPIO_IP only after verifying it is safe"
        record["gpio_candidates"] = candidates
        write_json(out_path, record)
        raise RuntimeError(record["reason"])
    # 可选安全开关：确认目标 GPIO 不会驱动危险外部电路后，才可设 L3_GPIO_ALLOW_NON_LED=1
    if "led" not in selected.lower() and os.environ.get("L3_GPIO_ALLOW_NON_LED") != "1":
        record["status"] = "NOT_RUN"
        record["claim_level"] = "not_measured"
        record["reason"] = "selected GPIO is not an LED path; refusing to drive it"
        record["selected_ip"] = selected
        write_json(out_path, record)
        raise RuntimeError(record["reason"])

    ip = overlay.ip_dict[selected]
    required = {selected: ip["type"]}
    verification = overlay_check.verify_overlay(
        overlay,
        info,
        required=required,
    )
    record["fclk0_hwh_mhz"] = info["clocks"].get("fclk0_mhz")
    record["fclk0_pynq_register_derived_mhz"] = getattr(
        Clocks(), "fclk0_mhz", None
    )
    check(record, "overlay_loaded", overlay.is_loaded())
    check(record, "gpio_present", selected in overlay.ip_dict)

    contract = json.loads(
        (repo / "skill/pynq-overlay-workflow/examples/axi_gpio.contract.json").read_text(
            encoding="utf-8"
        )
    )
    contract["addr_range"] = ip["addr_range"]
    contract["vlnv"] = ip["type"]
    contract["source"] = "PYNQ BaseOverlay hwh %s + AMD PG144" % art["hwh"]
    validated = regmap.load_contract(contract)
    hwh_registers = info["ips"].get(selected, {}).get("registers", {})
    record["hwh_cross_check"] = regmap.cross_check_hwh(validated, hwh_registers)

    params = ip.get("parameters", {})
    width_value = params.get("C_GPIO_WIDTH", params.get("c_gpio_width", 1))
    width = int(str(width_value), 0)
    width = max(1, min(width, 32))
    mask = (1 << width) - 1
    record["gpio_width"] = width

    dev = MMIO(ip["phys_addr"], ip["addr_range"])
    regs = regmap.RegisterMap(dev, validated)
    original_tri = regs.read("GPIO_TRI")
    original_data = regs.read("GPIO_DATA")
    record["original_tri"] = original_tri
    record["original_data"] = original_data
    record["selected_ip"] = selected
    record["verification"] = verification

    patterns = []
    for value in (0, mask, 0x55555555 & mask, 0xAAAAAAAA & mask):
        if value not in patterns:
            patterns.append(value)

    try:
        tri_value = original_tri & ~mask & 0xFFFFFFFF
        regs.write("GPIO_TRI", tri_value)
        tri_readback = regs.read("GPIO_TRI")
        record["register_trace"].append(
            {"register": "GPIO_TRI", "write": tri_value, "read": tri_readback}
        )
        check(record, "gpio_direction_readback", (tri_readback & mask) == 0)

        for value in patterns:
            regs.write("GPIO_DATA", value)
            time.sleep(0.03)
            actual = regs.read("GPIO_DATA") & mask
            record["register_trace"].append(
                {"register": "GPIO_DATA", "write": value, "read": actual}
            )
            check(record, "gpio_value_%02x" % value, actual == value)
    finally:
        regs.write("GPIO_DATA", original_data)
        regs.write("GPIO_TRI", original_tri)
        record["restored_data"] = regs.read("GPIO_DATA")
        record["restored_tri"] = regs.read("GPIO_TRI")
        check(
            record,
            "gpio_data_restored",
            (record["restored_data"] & mask) == (original_data & mask),
            record["restored_data"],
        )
        check(
            record,
            "gpio_direction_restored",
            (record["restored_tri"] & mask) == (original_tri & mask),
            record["restored_tri"],
        )

    bad_contract = json.loads(json.dumps(contract))
    bad_contract["registers"][0]["offset"] = "0x2"
    try:
        regmap.load_contract(bad_contract)
    except regmap.ContractError:
        check(record, "misaligned_contract_rejected", True)
    else:
        check(record, "misaligned_contract_rejected", False)

    record["status"] = "PASS"
except BaseException as exc:
    record["status"] = "FAIL" if record.get("status") != "NOT_RUN" else "NOT_RUN"
    record["error_type"] = type(exc).__name__
    record["error"] = str(exc)
    write_json(out_path, record)
    raise
else:
    write_json(out_path, record)
    print(json.dumps(record, indent=2, ensure_ascii=False))
    print("saved", out_path)